In [1]:
import numpy as np
import matplotlib.pyplot as plt

from easydynamics.job import Job
from easydynamics.experiment import Experiment
from easydynamics.experiment import Data
from easydynamics.analysis import Analysis

from easydynamics.sample import BrownianTranslationalDiffusion

from easydynamics.sample import SampleModel
from easydynamics.sample import LorentzianComponent
from easydynamics.sample import DeltaFunctionComponent
from easydynamics.sample import PolynomialComponent

from easydynamics.sample import GaussianComponent

from easydynamics.resolution import ResolutionHandler

from easyscience import Parameter

import scipp as sc

import plopp as pp

%matplotlib widget

In [2]:
# Create some fake data
Q=np.linspace(0.1,2,16)
E=np.linspace(-5,5,201)

diffusion_coefficients=[0.1,0.25,0.5,0.75]
convoluted_signal=np.zeros((len(Q),len(E)))

scale=0.7 #arbitrary scale factor for diffusion model


T=3

model=BrownianTranslationalDiffusion(name="DiffusionModel", diffusion_coefficient=diffusion_coefficients[T])
HWHM=model.calculate_width(Q)

QQISF=model.calculate_QISF(Q)
EISF=model.calculate_EISF(Q)

resolution=GaussianComponent(name="Resolution", area=1,width=0.1)

resolution_handler=ResolutionHandler()

sample_model=[]
for i in range(len(Q)):
    sample_model.append(SampleModel(name=f"SampleModel_{i}"))

    sample_model[i].add_component(DeltaFunctionComponent(area=scale*EISF[i]+0.23, name="Elastic"))
    sample_model[i].add_component(LorentzianComponent(area=scale*QQISF[i], name="QuasiElastic", width=HWHM[i]) )

    convoluted_signal[i,:] = resolution_handler.convolve(E,sample_model[i],resolution)+0.05+0.01*np.random.normal(size=len(E))



Q_scipp=sc.array(dims=['Q'],values=Q, unit='1/angstrom')
E_scipp=sc.array(dims=['energy'],values=E,unit='meV')
intensity_scipp=sc.array(dims=['Q','energy'],values=convoluted_signal,variances=0.01*convoluted_signal)

diffusion_data = sc.DataArray(data=intensity_scipp, coords={'Q':Q_scipp,'energy': E_scipp})


# pp.slicer(diffusion_data.transpose(),coords=['energy','Q'],keep=['energy'])



In [3]:
pp.slicer(diffusion_data.transpose(),coords=['energy','Q'],keep=['energy'])


InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…

In [4]:


diffusion_job= Job(name='BrownianDiffusion')


exp=Experiment()
data=Data()
data.append(diffusion_data)

exp.set_data(data)

diffusion_job.set_experiment(exp)
diffusion_job.generate_empty_analysis_array()


bg=SampleModel('Background')
bg.add_component(PolynomialComponent(coefficients=[0.5]))
diffusion_job.set_background_model(bg)
diffusion_job.set_background_model_for_all_analyses()

resolution=SampleModel()
resolution.add_component(GaussianComponent(name="Resolution", area=1,width=0.1))
diffusion_job.set_resolution_model(resolution)
diffusion_job.set_resolution_model_for_all_analyses()




diffusion_model=BrownianTranslationalDiffusion(name="DiffusionModel", diffusion_coefficient=0.3,scale=1.0)
diffusion_job.set_diffusion_model(diffusion_model)
# diffusion_job.set_theory_for_all_analyses(diffusion_model)



# delta_model=SampleModel(name="DeltaModel")
# delta_model.add_component(DeltaFunctionComponent(name="Delta",area=1.0))
# diffusion_job.set_theory_for_all_analyses(delta_model)

# delta_model=SampleModel(name="DeltaModel")
delta_model=DeltaFunctionComponent(name="Delta",area=0.2)
diffusion_job.set_theory_for_all_analyses(delta_model)


In [5]:
diffusion_job._analysis

[Analysis `Analysis(0,)`,
 Analysis `Analysis(1,)`,
 Analysis `Analysis(2,)`,
 Analysis `Analysis(3,)`,
 Analysis `Analysis(4,)`,
 Analysis `Analysis(5,)`,
 Analysis `Analysis(6,)`,
 Analysis `Analysis(7,)`,
 Analysis `Analysis(8,)`,
 Analysis `Analysis(9,)`,
 Analysis `Analysis(10,)`,
 Analysis `Analysis(11,)`,
 Analysis `Analysis(12,)`,
 Analysis `Analysis(13,)`,
 Analysis `Analysis(14,)`,
 Analysis `Analysis(15,)`]

In [6]:
diffusion_job.analysis[5].get_parameters()

[<Parameter 'scale': 1.0000, bounds=[-inf:inf]>,
 <Parameter 'Lorentzian center': 0.0000 meV (fixed), bounds=[-inf:inf]>,
 <Parameter 'Gamma': 0.1613, bounds=[-inf:inf]>,
 <Parameter 'Delta area': 0.2000 meV, bounds=[0.0:inf]>,
 <Parameter 'Delta center': 0.0000 meV (fixed), bounds=[-inf:inf]>,
 <Parameter 'diffusion_coefficient': 0.3000, bounds=[-inf:inf]>,
 <Parameter 'scale': 1.0000, bounds=[-inf:inf]>,
 <Parameter 'Resolution area': 1.0000 meV (fixed), bounds=[0.0:inf]>,
 <Parameter 'Resolution center': 0.0000 meV (fixed), bounds=[-inf:inf]>,
 <Parameter 'Resolution width': 0.1000 meV (fixed), bounds=[0.0:inf]>,
 <Parameter 'Polynomial_c0': 0.5000, bounds=[-inf:inf]>,
 <Parameter 'offset': 0.0000 meV, bounds=[-inf:inf]>]

In [7]:
diffusion_job.analysis[5].get_fit_parameters()

[<Parameter 'scale': 1.0000, bounds=[-inf:inf]>,
 <Parameter 'Delta area': 0.2000 meV, bounds=[0.0:inf]>,
 <Parameter 'diffusion_coefficient': 0.3000, bounds=[-inf:inf]>,
 <Parameter 'scale': 1.0000, bounds=[-inf:inf]>,
 <Parameter 'Polynomial_c0': 0.5000, bounds=[-inf:inf]>,
 <Parameter 'offset': 0.0000 meV, bounds=[-inf:inf]>]

In [8]:
diffusion_job.plot_data_and_model(intensity_min=0.0, intensity_max=4,
                            energy_min=-5, energy_max=5)

InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…

In [9]:
diffusion_job._analysis[0]._theory

diffusion_job._analysis[0]._experiment._data.data.values
diffusion_job._analysis[0]._experiment._data.data.coords['energy'].values

array([-5.  , -4.95, -4.9 , -4.85, -4.8 , -4.75, -4.7 , -4.65, -4.6 ,
       -4.55, -4.5 , -4.45, -4.4 , -4.35, -4.3 , -4.25, -4.2 , -4.15,
       -4.1 , -4.05, -4.  , -3.95, -3.9 , -3.85, -3.8 , -3.75, -3.7 ,
       -3.65, -3.6 , -3.55, -3.5 , -3.45, -3.4 , -3.35, -3.3 , -3.25,
       -3.2 , -3.15, -3.1 , -3.05, -3.  , -2.95, -2.9 , -2.85, -2.8 ,
       -2.75, -2.7 , -2.65, -2.6 , -2.55, -2.5 , -2.45, -2.4 , -2.35,
       -2.3 , -2.25, -2.2 , -2.15, -2.1 , -2.05, -2.  , -1.95, -1.9 ,
       -1.85, -1.8 , -1.75, -1.7 , -1.65, -1.6 , -1.55, -1.5 , -1.45,
       -1.4 , -1.35, -1.3 , -1.25, -1.2 , -1.15, -1.1 , -1.05, -1.  ,
       -0.95, -0.9 , -0.85, -0.8 , -0.75, -0.7 , -0.65, -0.6 , -0.55,
       -0.5 , -0.45, -0.4 , -0.35, -0.3 , -0.25, -0.2 , -0.15, -0.1 ,
       -0.05,  0.  ,  0.05,  0.1 ,  0.15,  0.2 ,  0.25,  0.3 ,  0.35,
        0.4 ,  0.45,  0.5 ,  0.55,  0.6 ,  0.65,  0.7 ,  0.75,  0.8 ,
        0.85,  0.9 ,  0.95,  1.  ,  1.05,  1.1 ,  1.15,  1.2 ,  1.25,
        1.3 ,  1.35,

In [10]:
result=diffusion_job.fit_simultaneous()


In [11]:

result

In [12]:
diffusion_job.plot_data_and_model(intensity_min=0.0, intensity_max=4,
                            energy_min=-5, energy_max=5)

InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…

In [13]:
diffusion_job.analysis[5].get_parameters()

[<Parameter 'scale': 0.7035 ± 0.0033, bounds=[-inf:inf]>,
 <Parameter 'Lorentzian center': 0.0000 meV (fixed), bounds=[-inf:inf]>,
 <Parameter 'Gamma': 0.4022 ± 0.0031, bounds=[-inf:inf]>,
 <Parameter 'Delta area': 0.2288 ± 0.0015 meV, bounds=[0.0:inf]>,
 <Parameter 'Delta center': 0.0000 meV (fixed), bounds=[-inf:inf]>,
 <Parameter 'diffusion_coefficient': 0.7479 ± 0.0058, bounds=[-inf:inf]>,
 <Parameter 'scale': 0.7035 ± 0.0033, bounds=[-inf:inf]>,
 <Parameter 'Resolution area': 1.0000 meV (fixed), bounds=[0.0:inf]>,
 <Parameter 'Resolution center': 0.0000 meV (fixed), bounds=[-inf:inf]>,
 <Parameter 'Resolution width': 0.1000 meV (fixed), bounds=[0.0:inf]>,
 <Parameter 'Polynomial_c0': 0.0493 ± 0.0007, bounds=[-inf:inf]>,
 <Parameter 'offset': -0.0009 ± 0.0024 meV, bounds=[-inf:inf]>]

In [14]:
diffusion_job.analysis[5].get_fit_parameters()

[<Parameter 'scale': 0.7035 ± 0.0033, bounds=[-inf:inf]>,
 <Parameter 'Delta area': 0.2288 ± 0.0015 meV, bounds=[0.0:inf]>,
 <Parameter 'diffusion_coefficient': 0.7479 ± 0.0058, bounds=[-inf:inf]>,
 <Parameter 'scale': 0.7035 ± 0.0033, bounds=[-inf:inf]>,
 <Parameter 'Polynomial_c0': 0.0493 ± 0.0007, bounds=[-inf:inf]>,
 <Parameter 'offset': -0.0009 ± 0.0024 meV, bounds=[-inf:inf]>]

In [15]:
diffusion_job.analysis[0].get_fit_parameters()

[<Parameter 'scale': 0.7035 ± 0.0033, bounds=[-inf:inf]>,
 <Parameter 'Delta area': 0.2288 ± 0.0015 meV, bounds=[0.0:inf]>,
 <Parameter 'diffusion_coefficient': 0.7479 ± 0.0058, bounds=[-inf:inf]>,
 <Parameter 'scale': 0.7035 ± 0.0033, bounds=[-inf:inf]>,
 <Parameter 'Polynomial_c0': 0.0488 ± 0.0006, bounds=[-inf:inf]>,
 <Parameter 'offset': 0.0003 ± 0.0009 meV, bounds=[-inf:inf]>]

In [23]:
diffusion_job_sequential= Job(name='BrownianDiffusionSequential')
diffusion_job_sequential.set_experiment(exp)
# diffusion_job_sequential.generate_empty_analysis_array()
diffusion_job_sequential.set_background_model(bg)
# diffusion_job_sequential.set_background_model_for_all_analyses()
diffusion_job_sequential.set_resolution_model(resolution)
# diffusion_job_sequential.set_resolution_model_for_all_analyses()

sequential_model=SampleModel(name="SequentialModel")
sequential_model.add_component(LorentzianComponent(name="QuasiElastic", area=1.0, width=0.5))
sequential_model.add_component(DeltaFunctionComponent(name="Elastic", area=0.2))
diffusion_job_sequential.set_theory(sequential_model)
diffusion_job_sequential.generate_analysis_for_cuts()

diffusion_job_sequential.plot_data_and_model(intensity_min=0.0, intensity_max=4,
                            energy_min=-5, energy_max=5)

InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…

In [24]:

diffusion_job_sequential.fit()

In [25]:
diffusion_job_sequential.plot_data_and_model(intensity_min=0.0, intensity_max=4,
                            energy_min=-5, energy_max=5)

InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…

In [26]:
diffusion_job_sequential.plot_fit_parameters("QuasiElastic width")

InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…